 Tutorial Part 1: Basic LangChain + Ollama Tracing with MLflow
 
 What you will learn:
   • How to connect MLflow to a local tracking server
   • How to enable automatic tracing for LangChain calls
   • How to run a simple LLM chain and inspect the trace on
     the MLflow UI

 Pre-requisites:
   • Ollama running locally  (ollama serve)
   • qwen3.5:0.8b model pulled (ollama pull qwen3.5:0.8b)
   • MLflow UI running on port 6001
       mlflow server --host 127.0.0.1 --port 6001

Start the server:

In [ ]:
import subprocess, sys
subprocess.Popen([
    sys.executable, "-m", "mlflow", "server",
    "--host", "127.0.0.1", "--port", "6001",
])

<Popen: returncode: None args: ['c:\\Users\\konan\\projects\\langchain-ollam...>

In [5]:
import mlflow
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate

1. Point MLflow at your local server using port 6001

In [6]:
mlflow.set_tracking_uri("http://127.0.0.1:6001")
mlflow.set_experiment("basic_tracing")

2026/10/24 19:03:49 INFO mlflow.tracking.fluent: Experiment with name 'basic_tracing' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1792886629534, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1792886629534, lifecycle_stage='active', name='basic_tracing', tags={}, trace_location=None, workspace='default'>

 2. Enable auto-tracing for every LangChain invocation

    This single line instruments all LangChain / LangGraph
    
    runnables automatically – no extra code needed

In [7]:
mlflow.langchain.autolog()

3. Check for Ollama available Model

In [ ]:
!ollama list

4. Build a minimal LangChain chain

In [14]:
llm = OllamaLLM(
    model="qwen3.6",
    base_url="http://localhost:11434",
    temperature=0.1,
)

prompt = PromptTemplate.from_template(
    "Answer the following question in one short sentence.\n\nQuestion: {question}"
)

chain = prompt | llm

5. Run inside an MLflow run so results are persisted

In [15]:
with mlflow.start_run(run_name="basic-chain-demo"):
    questions = [
        "What is the capital of France?",
        "Who invented the telephone?",
        "What is 15 multiplied by 7?",
    ]

    for q in questions:
        answer = chain.invoke({"question": q})
        print(f"\nQ: {q}")
        print(f"A: {answer}")

print("\n✅  Run complete – open http://localhost:6001 to view traces.")


Q: What is the capital of France?
A: The capital of France is Paris.

Q: Who invented the telephone?
A: Alexander Graham Bell invented the telephone.

Q: What is 15 multiplied by 7?
A: 15 multiplied by 7 equals 105.
🏃 View run basic-chain-demo at: http://127.0.0.1:6001/#/experiments/1/runs/e2ff46ad5efe48ab915a8c1755bb4781
🧪 View experiment at: http://127.0.0.1:6001/#/experiments/1

✅  Run complete – open http://localhost:6001 to view traces.


[Trace(trace_id='tr-d5c8a44bff1b2ddb63cde52080369d88'), Trace(trace_id='tr-26c2c68ee544a26424d8e9cc6475378d')]